<a href="https://colab.research.google.com/github/1021114Carlos/MIT_MM_Finance/blob/Finance-shop/Courses/Derivative_Markets/M9_Credit_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Library

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

Q1. A)

In [ ]:
nav = 200000*38.00
nav

7600000.0

Q2) A, B, C sections

In [ ]:
def solve_bond_protection():
    print("--- Bond and Credit Default Option Solver ---")

    # User Inputs
    face_value = float(input("Enter Face Value of the bond (e.g., 1000): "))
    bond_price = float(input("Enter current Price of the risky bond (e.g., 935.914184): "))
    expected_return = float(input("Enter Expected Return as a decimal (e.g., 0.0429): "))
    prob_default = float(input("Enter Probability of Default as a decimal (e.g., 0.03): "))
    treasury_price = float(input("Enter Price of the 1-year Zero-Coupon Treasury (e.g., 946.282041): "))
    protection_notional = float(input("Enter Face Value for protection calculation (e.g., 100000): "))

    # Part A: Implied Recovery Amount
    # Expected CF = Bond Price * (1 + Expected Return)
    expected_cf = bond_price * (1 + expected_return)

    # Expected CF = (Prob No Default * Face Value) + (Prob Default * Recovery)
    # Recovery = (Expected CF - (1 - Prob Default) * Face Value) / Prob Default
    prob_no_default = 1 - prob_default
    recovery_value = (expected_cf - (prob_no_default * face_value)) / prob_default

    # Part B: Portfolio Cash Flows (per unit of Face Value)
    # Portfolio: Long 1 Risky Bond, Short 1 Treasury
    t0_net_cf = treasury_price - bond_price

    # Time 1 Cash Flows
    t1_no_default = face_value - face_value  # (Bond In - Treasury Out)
    t1_default = recovery_value - face_value # (Recovery In - Treasury Out)

    # Part C: Upfront Fee for Protection (No-Arbitrage)
    # Fee is the difference in price scaled to the protection notional
    fee_per_unit = (treasury_price - bond_price) / face_value
    total_upfront_fee = fee_per_unit * protection_notional

    # Output Results
    print("\n--- RESULTS ---")
    print(f"A) Implied Recovery Amount per {face_value}: {recovery_value:.6f}")

    print(f"\nB) Net Cash Flows (Portfolio of Long Bond/Short Treasury):")
    print(f"   Time 0 Net CF: {t0_net_cf:+.6f}")
    print(f"   Time 1 (No Default): {t1_no_default:.6f}")
    print(f"   Time 1 (Default):    {t1_default:.6f}")

    print(f"\nC) Upfront Fee for {protection_notional:,.2f} Notional:")
    print(f"   Total Fee: {total_upfront_fee:.6f}")

if __name__ == "__main__":
    solve_bond_protection()

--- Bond and Credit Default Option Solver ---
Enter Face Value of the bond (e.g., 1000): 1000
Enter current Price of the risky bond (e.g., 935.914184): 935.9141
Enter Expected Return as a decimal (e.g., 0.0429): 0.0429
Enter Probability of Default as a decimal (e.g., 0.03): 0.03
Enter Price of the 1-year Zero-Coupon Treasury (e.g., 946.282041): 946.2820
Enter Face Value for protection calculation (e.g., 100000): 100000

--- RESULTS ---
A) Implied Recovery Amount per 1000.0: 202.160496

B) Net Cash Flows (Portfolio of Long Bond/Short Treasury):
   Time 0 Net CF: +10.367900
   Time 1 (No Default): 0.000000
   Time 1 (Default):    -797.839504

C) Upfront Fee for 100,000.00 Notional:
   Total Fee: 1036.790000


Q3:

In [ ]:
def calculate_merton_layers():
    # --- User Inputs ---
    print("--- Merton Model Junior Debt Calculator ---")
    V = float(input("Current Asset Value (M): ") or 178)
    vol = float(input("Asset Volatility (e.g., 0.14): ") or 0.14)
    r = float(input("Risk-free Rate (e.g., 0.025): ") or 0.025)
    T = float(input("Time to Maturity in Years: ") or 5)
    K_senior = float(input("Face Value of Senior Debt (M): ") or 100)
    K_junior = float(input("Face Value of Junior Debt (M): ") or 40)

    K_total = K_senior + K_junior

    def black_scholes_call(S, K, T, r, sigma):
        if T <= 0: return max(0, S - K)
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        call_val = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        return call_val, d1, d2

    # Step 1: Calculate Equity (Call option on total debt)
    equity, _, _ = black_scholes_call(V, K_total, T, r, vol)

    # Step 2: Calculate Senior Debt Value
    # Value of Senior Debt = Assets - Call Option with K = Senior Face Value
    call_snr, _, _ = black_scholes_call(V, K_senior, T, r, vol)
    senior_debt_val = V - call_snr

    # Step 3: Calculate Junior Debt Value
    # Junior Debt = Total Assets - Equity - Senior Debt
    junior_debt_val = V - equity - senior_debt_val

    # --- Output Results ---
    print(f"\n--- Results (6 Decimal Places) ---")
    print(f"Equity Value:      {equity:12.6f} M")
    print(f"Senior Debt Value: {senior_debt_val:12.6f} M")
    print(f"Junior Debt Value: {junior_debt_val:12.6f} M")
    print(f"Total Asset Value: {V:12.6f} M")

    # --- Visualization ---
    labels = ['Senior Debt', 'Junior Debt', 'Equity']
    values = [senior_debt_val, junior_debt_val, equity]
    colors = ['#2E4053', '#A93226', '#239B56']

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, edgecolor='black', alpha=0.8)

    # Add data labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}M',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')

    plt.title(f'Capital Structure Valuation (Merton Model)\nAssets=${V}M, Vol={vol*100}%, T={T}yrs', fontsize=14)
    plt.ylabel('Market Value (Millions)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Save the plot
    plt.savefig('merton_valuation.png')
    print("\nVisualization saved as 'merton_valuation.png'.")
    plt.show()

if __name__ == "__main__":
    calculate_merton_layers()